# 官方 OmniDocBench end2end 评测冒烟（锁定 commit）

3 页固定子集（seed 42）→ 基线推理（CPU）→ 官方 `pdf_validation.py --config` 出分。


In [ ]:
# 自包含引导：把仓库代码快照写入 /kaggle/working
import json, os, sys
from pathlib import Path

FILES = {"src/__init__.py": "\"\"\"MV-AI Lab Document Intelligence Experimental Framework.\n\nPhase 2 核心模块：config / data / model / prompts / inference / evaluation / visualization。\n设计依据见 docs/notebook-design.md。\n\"\"\"\n\n__version__ = \"0.2.0\"\n\n", "src/config.py": "\"\"\"配置加载与路径解析。\n\nKaggle 环境规则（见 docs/notebook-design.md §6）：\n- 代码不硬编码绝对路径；\n- /kaggle/input 只读，/kaggle/working 是工作目录；\n- 所有实验产物写在工作目录下的 results/，便于会话结束后导出；\n- 环境变量 ODB_PROJECT_ROOT 可覆盖工作目录。\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional\n\nimport yaml\n\n_DEFAULT_CONFIG = Path(__file__).resolve().parents[1] / \"configs\" / \"default.yaml\"\n\n\ndef load_config(path: Optional[os.PathLike] = None) -> Dict[str, Any]:\n    \"\"\"加载 YAML 配置；默认使用仓库内 configs/default.yaml。\"\"\"\n    cfg_path = Path(path) if path is not None else _DEFAULT_CONFIG\n    with open(cfg_path, \"r\", encoding=\"utf-8\") as f:\n        cfg = yaml.safe_load(f)\n    return cfg if isinstance(cfg, dict) else {}\n\n\ndef project_root() -> Path:\n    \"\"\"实验工作目录（Kaggle 上为 /kaggle/working，本地为当前目录）。\"\"\"\n    root = os.environ.get(\"ODB_PROJECT_ROOT\")\n    return Path(root).resolve() if root else Path.cwd().resolve()\n\n\ndef repo_root() -> Path:\n    \"\"\"本仓库根目录（定位 src/、configs/、prompts/）。\"\"\"\n    return Path(__file__).resolve().parents[1]\n\n\ndef results_dir(config: Optional[Dict[str, Any]] = None) -> Path:\n    \"\"\"实验结果根目录（默认 <project_root>/results）。\"\"\"\n    cfg = config if config is not None else load_config()\n    rel = cfg.get(\"paths\", {}).get(\"results_dir\", \"results\")\n    return project_root() / rel\n\n", "src/data.py": "\"\"\"OmniDocBench 数据加载、统计与采样。\n\n原则（docs/notebook-design.md §9）：\n- 官方 JSON 只读，绝不修改；\n- 官方 1651 页是 Benchmark 数据，没有官方 SFT train split；\n- 教学训练子集（Phase 3）只从 v1.5 子集页面抽取，并标记 NOT for official claims；\n- 数据本体不入 Git，只记录 revision 与 manifest。\n\"\"\"\n\nfrom __future__ import annotations\n\nimport collections\nimport hashlib\nimport json\nimport os\nimport random\nimport shutil\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Optional, Sequence\n\nDATASET_ID = \"opendatalab/OmniDocBench\"\nDATASET_REVISION = \"aa1ee96d106dbe53d0ae59474d75c6e6d9b53fec\"\nANNOTATION_FILENAME = \"OmniDocBench.json\"\nIMAGES_DIRNAME = \"images\"\n\n\ndef find_dataset_root(extra_dirs: Optional[Iterable[os.PathLike]] = None) -> Path:\n    \"\"\"按顺序查找 OmniDocBench 数据根目录（含 OmniDocBench.json 的那一层）。\n\n    顺序：环境变量 OMNIDOCBENCH_DIR → /kaggle/input 下所有挂载 → 当前目录\n    data/OmniDocBench → 调用方传入的额外目录。\n    \"\"\"\n    candidates: List[Path] = []\n    env = os.environ.get(\"OMNIDOCBENCH_DIR\")\n    if env:\n        candidates.append(Path(env))\n    kaggle_input = Path(\"/kaggle/input\")\n    if kaggle_input.exists():\n        candidates.extend(sorted(p for p in kaggle_input.iterdir() if p.is_dir()))\n    candidates.append(Path.cwd() / \"data\" / \"OmniDocBench\")\n    for extra in extra_dirs or []:\n        candidates.append(Path(extra))\n\n    for cand in candidates:\n        cand = Path(cand)\n        if (cand / ANNOTATION_FILENAME).is_file():\n            return cand.resolve()\n        sub = cand / \"OmniDocBench\"\n        if (sub / ANNOTATION_FILENAME).is_file():\n            return sub.resolve()\n    raise FileNotFoundError(\n        \"未找到 OmniDocBench 数据目录。请设置环境变量 OMNIDOCBENCH_DIR，\"\n        \"或在 Kaggle 上添加官方数据集到 /kaggle/input，或运行 \"\n        \"src.data.download_dataset() 下载到 <project_root>/data/OmniDocBench。\"\n    )\n\n\ndef download_dataset(\n    revision: str = DATASET_REVISION,\n    target_dir: Optional[os.PathLike] = None,\n    allow_patterns: Optional[Sequence[str]] = None,\n) -> Path:\n    \"\"\"从 Hugging Face 下载官方数据集（图片 + 标注 JSON，不含展示大图）。\n\n    ⚠️ 数据集仅限研究用途、不可商用（官方 Copyright Statement）。\n    \"\"\"\n    from huggingface_hub import snapshot_download\n\n    target = Path(target_dir) if target_dir is not None else Path.cwd() / \"data\"\n    patterns = list(allow_patterns) if allow_patterns else [\n        \"images/*\",\n        \"OmniDocBench.json\",\n        \"README.md\",\n        \"README_ZH.md\",\n    ]\n    local = snapshot_download(\n        repo_id=DATASET_ID,\n        repo_type=\"dataset\",\n        revision=revision,\n        local_dir=target / \"OmniDocBench\",\n        allow_patterns=patterns,\n    )\n    local_path = Path(local)\n    # huggingface_hub 会在 local_dir 内生成 .cache 元数据；清理它，\n    # 避免 Kaggle 输出包把缓存一并打包（体积膨胀 + Windows 长路径问题）。\n    cache_dir = local_path / \".cache\"\n    if cache_dir.exists():\n        shutil.rmtree(cache_dir, ignore_errors=True)\n    return local_path\n\n\ndef load_annotations(root: os.PathLike) -> List[Dict[str, Any]]:\n    \"\"\"加载官方标注 JSON（只读）。\"\"\"\n    path = Path(root) / ANNOTATION_FILENAME\n    with open(path, \"r\", encoding=\"utf-8\") as f:\n        annotations = json.load(f)\n    if not isinstance(annotations, list):\n        raise ValueError(\"OmniDocBench.json 顶层结构不是列表，请检查数据版本\")\n    return annotations\n\n\ndef page_image_path(root: os.PathLike, page: Dict[str, Any]) -> Path:\n    rel = page[\"page_info\"][\"image_path\"]\n    return Path(root) / IMAGES_DIRNAME / Path(rel).name\n\n\ndef load_page_image(root: os.PathLike, page: Dict[str, Any]) -> Any:\n    \"\"\"加载页面图像为 PIL RGB Image。\"\"\"\n    from PIL import Image\n\n    path = page_image_path(root, page)\n    with Image.open(path) as img:\n        return img.convert(\"RGB\")\n\n\ndef sample_id(page: Dict[str, Any]) -> str:\n    \"\"\"稳定样本 ID：图像文件名去扩展名（官方文件名本身是 UUID）。\"\"\"\n    return Path(page[\"page_info\"][\"image_path\"]).stem\n\n\ndef page_attribute(page: Dict[str, Any]) -> Dict[str, Any]:\n    return page.get(\"page_info\", {}).get(\"page_attribute\", {}) or {}\n\n\ndef subset_of(page: Dict[str, Any]) -> str:\n    \"\"\"官方子集标记：v1.5 / equation_hard / layout_hard / table_hard。\"\"\"\n    value = page_attribute(page).get(\"subset\")\n    if isinstance(value, str):\n        return value\n    if isinstance(value, (list, tuple)) and value:\n        return \"|\".join(str(v) for v in value)\n    return \"\"\n\n\ndef select_pages(\n    annotations: Sequence[Dict[str, Any]],\n    n: int,\n    seed: int = 42,\n    teaching_only: bool = False,\n) -> List[Dict[str, Any]]:\n    \"\"\"分层抽样 n 页（按 data_source 轮转），确定性可复现。\n\n    teaching_only=True：只从 v1.5 子集页面抽取（教学训练子集，Phase 3 使用）。\n    Baseline 推理默认 teaching_only=False（用官方页面做 zero-shot 评测是允许的，\n    禁止的是用官方页面训练后再在官方集合报成绩）。\n    \"\"\"\n    rng = random.Random(seed)\n    pool = list(annotations)\n    if teaching_only:\n        pool = [p for p in pool if subset_of(p) == \"v1.5\"]\n    strata: Dict[str, List[int]] = {}\n    for i, p in enumerate(pool):\n        key = page_attribute(p).get(\"data_source\", \"unknown\")\n        strata.setdefault(key, []).append(i)\n    picked: List[int] = []\n    keys = sorted(strata)\n    while len(picked) < n and any(strata[k] for k in keys):\n        for k in keys:\n            if not strata[k]:\n                continue\n            idx = strata[k].pop()\n            if idx not in picked:\n                picked.append(idx)\n            if len(picked) >= n:\n                break\n    if len(picked) < n:\n        # 池子不足时补足（理论上只在 teaching_only 且 n 过大时出现）\n        picked = sorted(picked)\n    rng.shuffle(picked)\n    return [pool[i] for i in picked[:n]]\n\n\ndef build_stats(annotations: Sequence[Dict[str, Any]]) -> Dict[str, Dict[str, int]]:\n    \"\"\"数据分布统计：与设计文档 §4.2 的官方数字可对照。\"\"\"\n    doc_type: Dict[str, int] = collections.Counter()\n    language: Dict[str, int] = collections.Counter()\n    layout: Dict[str, int] = collections.Counter()\n    subset: Dict[str, int] = collections.Counter()\n    block_cat: Dict[str, int] = collections.Counter()\n    n_tables = 0\n    n_tables_with_html = 0\n    n_formulas = 0\n    n_formulas_with_latex = 0\n    relations: Dict[str, int] = collections.Counter()\n\n    for page in annotations:\n        attr = page_attribute(page)\n        doc_type[attr.get(\"data_source\", \"unknown\")] += 1\n        language[attr.get(\"language\", \"unknown\")] += 1\n        layout[attr.get(\"layout\", \"unknown\")] += 1\n        subset[subset_of(page) or \"unknown\"] += 1\n        for det in page.get(\"layout_dets\", []) or []:\n            cat = det.get(\"category_type\", \"unknown\")\n            block_cat[cat] += 1\n            if cat == \"table\":\n                n_tables += 1\n                if det.get(\"html\"):\n                    n_tables_with_html += 1\n            if cat in (\"equation_isolated\", \"equation_caption\", \"equation_semantic\", \"equation_explanation\"):\n                n_formulas += 1\n                if det.get(\"latex\"):\n                    n_formulas_with_latex += 1\n        for rel in (page.get(\"extra\") or {}).get(\"relation\", []) or []:\n            relations[rel.get(\"relation\", rel.get(\"relation_type\", \"unknown\"))] += 1\n\n    return {\n        \"pages\": {\"total\": len(list(annotations))},\n        \"document_type\": dict(doc_type),\n        \"language\": dict(language),\n        \"layout\": dict(layout),\n        \"subset\": dict(subset),\n        \"block_category\": dict(block_cat),\n        \"table\": {\"count\": n_tables, \"with_html\": n_tables_with_html},\n        \"formula\": {\"count\": n_formulas, \"with_latex\": n_formulas_with_latex},\n        \"relation\": dict(relations),\n    }\n\n\ndef write_json(obj: Any, path: os.PathLike) -> Path:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with open(path, \"w\", encoding=\"utf-8\") as f:\n        json.dump(obj, f, ensure_ascii=False, indent=2)\n    return path\n\n\ndef read_json(path: os.PathLike) -> Any:\n    with open(path, \"r\", encoding=\"utf-8\") as f:\n        return json.load(f)\n\n\ndef sha256_file(path: os.PathLike) -> str:\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b\"\"):\n            h.update(chunk)\n    return h.hexdigest()\n\n\n# ---------------------------------------------------------------------------\n# Phase 3：教学训练子集（红线见 docs/notebook-design.md §9）\n# ---------------------------------------------------------------------------\n\nTEACHING_SUBSET_MARKER = (\n    \"OmniDocBench-derived teaching subset — NOT for official benchmark claims\"\n)\n\n\ndef build_teaching_split(\n    annotations: Sequence[Dict[str, Any]],\n    n_train: int = 24,\n    n_val: int = 8,\n    seed: int = 42,\n) -> Dict[str, List[Dict[str, Any]]]:\n    \"\"\"从 v1.5 子集页面构建教学 train/val 划分（分层、确定性）。\n\n    官方没有 SFT train split，本函数创建的是「教学子集」：\n    - 只取 subset == 'v1.5' 的页面，不触碰三个困难子集；\n    - 全部产物标记 TEACHING_SUBSET_MARKER；\n    - 不得用它在官方 1651 页上宣称成绩。\n    \"\"\"\n    pool = [p for p in annotations if subset_of(p) == \"v1.5\"]\n    val = select_pages(pool, n=n_val, seed=seed, teaching_only=True)\n    val_ids = {sample_id(p) for p in val}\n    rest = [p for p in pool if sample_id(p) not in val_ids]\n    train = select_pages(rest, n=n_train, seed=seed, teaching_only=True)\n    return {\"train\": train, \"val\": val, \"marker\": TEACHING_SUBSET_MARKER}\n\n\ndef write_split_manifest(split: Dict[str, Any], out_dir: os.PathLike) -> Path:\n    \"\"\"把 train/val 页面写为 manifest（image_id、属性、来源子集）。\"\"\"\n    out = Path(out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    for name in (\"train\", \"val\"):\n        rows = []\n        for page in split[name]:\n            attr = page_attribute(page)\n            rows.append(\n                {\n                    \"image_id\": sample_id(page),\n                    \"image_path\": page[\"page_info\"][\"image_path\"],\n                    \"subset\": subset_of(page),\n                    \"document_type\": attr.get(\"data_source\", \"unknown\"),\n                    \"language\": attr.get(\"language\", \"unknown\"),\n                    \"layout\": attr.get(\"layout\", \"unknown\"),\n                    \"marker\": split.get(\"marker\", TEACHING_SUBSET_MARKER),\n                }\n            )\n        write_json(rows, out / f\"{name}.jsonl\")\n    write_json({\"marker\": split.get(\"marker\", TEACHING_SUBSET_MARKER)}, out / \"split_info.json\")\n    return out\n\n\n# layout_dets 类别 → docling item 的近似映射（教学用途）。\n# ⚠️ 28 类官方标注与 Docling item 并非一一对应；表格/公式的完整结构转换\n# 是 Notebook 05 的一个开放练习，首次运行时记录实际行为。\n_CATEGORY_TO_DOCLING = {\n    \"title\": \"heading\",\n    \"text_block\": \"text\",\n    \"list_group\": \"text\",\n    \"reference\": \"text\",\n    \"figure\": \"picture\",\n    \"figure_caption\": \"text\",\n    \"table_caption\": \"text\",\n    \"table_footnote\": \"text\",\n    \"figure_footnote\": \"text\",\n    \"page_footnote\": \"text\",\n    \"code_txt\": \"text\",\n}\n\n\ndef page_to_doctags(page: Dict[str, Any]) -> Dict[str, Any]:\n    \"\"\"把官方 layout_dets 转换为教学用 DocTags 目标（近似转换）。\n\n    返回 {\"doctags\": str, \"api_note\": str, \"skipped\": [category...]}。\n    表格与公式的完整结构暂不转换（见模块 docstring 的说明），\n    其文本内容按 reading order 以 text 形式保留，保证训练样本不丢文字。\n    \"\"\"\n    api_note = \"unknown\"\n    try:\n        from docling_core.types.doc import DoclingDocument\n        from docling_core.types.doc.labels import DocItemLabel\n    except ImportError as exc:\n        raise ImportError(\n            \"缺少 docling-core，无法把 GT 转换为 DocTags。\"\n            \"请在 Kaggle 上安装 docling-core。\"\n        ) from exc\n\n    doc = DoclingDocument(name=sample_id(page))\n    skipped: List[str] = []\n    for det in sorted(\n        (d for d in page.get(\"layout_dets\", []) or [] if not d.get(\"ignore\")),\n        key=lambda d: d.get(\"order\", 0),\n    ):\n        cat = det.get(\"category_type\", \"unknown\")\n        text = det.get(\"text\") or \"\"\n        mapping = _CATEGORY_TO_DOCLING.get(cat)\n        if mapping == \"heading\" and text:\n            try:\n                doc.add_heading(text=text, level=1)\n            except Exception as exc:  # noqa: BLE001\n                api_note = f\"add_heading 失败：{exc}\"\n                skipped.append(cat)\n        elif mapping == \"text\" and text:\n            try:\n                doc.add_text(label=DocItemLabel.TEXT, text=text)\n            except Exception as exc:  # noqa: BLE001\n                api_note = f\"add_text 失败：{exc}\"\n                skipped.append(cat)\n        elif mapping == \"picture\":\n            try:\n                doc.add_picture(prov=None)\n            except Exception as exc:  # noqa: BLE001\n                api_note = f\"add_picture 失败：{exc}\"\n                skipped.append(cat)\n        else:\n            skipped.append(cat)\n    try:\n        doctags = doc.export_to_doctags()\n        api_note = f\"{api_note} | export_to_doctags OK\"\n    except AttributeError:\n        doctags = doc.export_to_document_tokens()\n        api_note = f\"{api_note} | export_to_document_tokens OK\"\n    return {\"doctags\": doctags, \"api_note\": api_note, \"skipped\": sorted(set(skipped))}\n\n\ndef build_sft_records(\n    pages: Sequence[Dict[str, Any]],\n    root: os.PathLike,\n    prompt_text: str,\n) -> List[Dict[str, str]]:\n    \"\"\"构造 SFT 记录：图像路径 + 指令 + GT 派生 DocTags 目标。\"\"\"\n    records = []\n    for page in pages:\n        converted = page_to_doctags(page)\n        records.append(\n            {\n                \"image_id\": sample_id(page),\n                \"image_path\": str(page_image_path(root, page)),\n                \"instruction\": prompt_text,\n                \"target_doctags\": converted[\"doctags\"],\n                \"api_note\": converted[\"api_note\"],\n                \"skipped_categories\": \",\".join(converted[\"skipped\"]),\n            }\n        )\n    return records\n", "src/model.py": "\"\"\"DocumentModelAdapter 抽象与 SmolDocling 实现。\n\n设计目标（docs/notebook-design.md §8.4）：Benchmark 框架与模型解耦。\n- Notebook / scripts 只依赖 DocumentModelAdapter 接口；\n- 未来可增加 QwenVLAdapter、PaddleOCRVLAdapter，评测框架不变。\n\nSmolDocling 官方接口（模型卡，revision ce51f56c…，2025-09-17）：\n- AutoProcessor + AutoModelForVision2Seq\n- bf16；CUDA 时优先 flash_attention_2，失败自动回退 eager\n- max_new_tokens=8192\n- DocTags → DoclingDocument 转换在 doctags_to_docling() 中做了新旧 API 双路径兼容\n  （风险 R2），并记录实际生效的 API 路径。\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport time\nimport warnings\nfrom abc import ABC, abstractmethod\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nOFFICIAL_MODEL_ID = \"docling-project/SmolDocling-256M-preview\"\nOFFICIAL_MODEL_REVISION = \"ce51f56c4ebe36e0b1c3a55f67b261ba22a50bf8\"\nDEFAULT_PROMPT = \"Convert this page to docling.\"\n\n\nclass DocumentModelAdapter(ABC):\n    \"\"\"模型适配器统一接口。\"\"\"\n\n    def __init__(\n        self,\n        model_id: str = OFFICIAL_MODEL_ID,\n        revision: Optional[str] = OFFICIAL_MODEL_REVISION,\n    ) -> None:\n        self.model_id = model_id\n        self.revision = revision\n\n    @abstractmethod\n    def load(self, device: str = \"auto\", **kwargs: Any) -> \"DocumentModelAdapter\":\n        \"\"\"加载模型与 processor，返回 self 以便链式调用。\"\"\"\n\n    @abstractmethod\n    def predict(\n        self,\n        image: Any,\n        prompt: str = DEFAULT_PROMPT,\n        **generation_kwargs: Any,\n    ) -> Dict[str, Any]:\n        \"\"\"对单页图像生成结构化输出。\n\n        返回字段：doctags / prompt / latency_sec / generation_config /\n        model_id / model_revision / device。\n        \"\"\"\n\n    def save_prediction(self, prediction: Dict[str, Any], out_path: Path) -> None:\n        out_path = Path(out_path)\n        out_path.parent.mkdir(parents=True, exist_ok=True)\n        with open(out_path, \"w\", encoding=\"utf-8\") as f:\n            json.dump(prediction, f, ensure_ascii=False, indent=2)\n\n\nclass SmolDoclingAdapter(DocumentModelAdapter):\n    \"\"\"SmolDocling-256M-preview 适配器（transformers 路径）。\"\"\"\n\n    def __init__(\n        self,\n        model_id: str = OFFICIAL_MODEL_ID,\n        revision: Optional[str] = OFFICIAL_MODEL_REVISION,\n    ) -> None:\n        super().__init__(model_id=model_id, revision=revision)\n        self.model: Any = None\n        self.processor: Any = None\n        self.device: str = \"cpu\"\n        self.dtype_name: str = \"float32\"\n        self.model_class: str = \"\"\n        self.attn_implementation: str = \"\"\n        self.device_fallback_reason: str = \"\"\n\n    def load(\n        self,\n        device: str = \"auto\",\n        dtype: str = \"auto\",\n        attention: str = \"auto\",\n        **kwargs: Any,\n    ) -> \"SmolDoclingAdapter\":\n        \"\"\"加载模型。\n\n        dtype=auto：CUDA 支持 bf16 → bf16；CUDA → fp16；CPU → fp32。\n        attention=auto：CUDA 优先 flash_attention_2，失败回退 eager。\n        \"\"\"\n        import torch\n        from transformers import AutoProcessor\n\n        if device == \"auto\":\n            device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n        self.device = device\n\n        self.processor = AutoProcessor.from_pretrained(\n            self.model_id, revision=self.revision\n        )\n\n        if dtype == \"auto\":\n            if device == \"cuda\":\n                dtype = \"bfloat16\" if torch.cuda.is_bf16_supported() else \"float16\"\n            else:\n                dtype = \"float32\"\n        self.dtype_name = dtype\n        torch_dtype = getattr(torch, dtype)\n\n        attn = attention\n        if attn == \"auto\":\n            attn = \"flash_attention_2\" if device == \"cuda\" else \"eager\"\n\n        def _load(model_cls: Any, torch_dtype: Any, attn_impl: str) -> Any:\n            return model_cls.from_pretrained(\n                self.model_id,\n                revision=self.revision,\n                torch_dtype=torch_dtype,\n                _attn_implementation=attn_impl,\n            )\n\n        # 官方模型卡用 AutoModelForVision2Seq；新版 transformers 可能仅保留\n        # AutoModelForImageTextToText（类名合并）。对「模型类 × 注意力实现」\n        # 做全组合回退，并记录实际生效的组合（设计文档风险 R3）。\n        model_classes: List[Tuple[str, Any]] = []\n        try:\n            from transformers import AutoModelForVision2Seq\n\n            model_classes.append((\"AutoModelForVision2Seq\", AutoModelForVision2Seq))\n        except ImportError:\n            pass\n        try:\n            from transformers import AutoModelForImageTextToText\n\n            model_classes.append(\n                (\"AutoModelForImageTextToText\", AutoModelForImageTextToText)\n            )\n        except ImportError:\n            pass\n        if not model_classes:\n            raise ImportError(\n                \"transformers 中找不到 AutoModelForVision2Seq / \"\n                \"AutoModelForImageTextToText，请检查 transformers 版本。\"\n            )\n\n        last_error: Optional[Exception] = None\n        attn_candidates = [attn, \"eager\"] if attn != \"eager\" else [\"eager\"]\n        for name, model_cls in model_classes:\n            for attn_impl in attn_candidates:\n                try:\n                    self.model = _load(model_cls, torch_dtype, attn_impl)\n                    self.model_class = name\n                    self.attn_implementation = attn_impl\n                    break\n                except Exception as exc:  # noqa: BLE001\n                    last_error = exc\n                    warnings.warn(f\"{name} + {attn_impl} 加载失败：{exc}\")\n            if self.model is not None:\n                break\n        if self.model is None:\n            raise RuntimeError(\n                \"SmolDocling 加载失败（已尝试全部模型类/注意力组合）。\"\n                f\"最后错误：{last_error}\"\n            )\n        self.model.to(self.device).eval()\n        # CUDA 可用性自检：部分 Kaggle GPU（如 P100/sm_60）与镜像自带的新版\n        # torch 不兼容（no kernel image），任何 CUDA 算子都会报 AcceleratorError。\n        # 检测到后自动回退 CPU 并记录原因（设计文档风险 R6/R3）。\n        if self.device == \"cuda\":\n            try:\n                probe = torch.zeros((2, 2), device=\"cuda\")\n                (probe @ probe).sum().item()\n            except Exception as exc:  # noqa: BLE001\n                self.device_fallback_reason = (\n                    f\"CUDA kernel 不可用（{type(exc).__name__}: {exc}），\"\n                    \"自动回退 CPU\"\n                )\n                warnings.warn(self.device_fallback_reason)\n                self.device = \"cpu\"\n                self.dtype_name = \"float32\"\n                # 顺序很重要：先把权重搬到 CPU（纯内存拷贝，不需要 CUDA kernel），\n                # 再在 CPU 上转 float。若先在故障 GPU 上转 float 会再次触发\n                # \"no kernel image\" 错误。\n                self.model = self.model.to(\"cpu\").float()\n                probe_cpu = torch.zeros((2, 2))\n                (probe_cpu @ probe_cpu).sum().item()  # CPU 自检\n        return self\n\n    def build_messages(self, prompt: str) -> List[Dict[str, Any]]:\n        \"\"\"官方模型卡的消息格式（image 占位 + 文本指令）。\"\"\"\n        return [\n            {\n                \"role\": \"user\",\n                \"content\": [\n                    {\"type\": \"image\"},\n                    {\"type\": \"text\", \"text\": prompt},\n                ],\n            }\n        ]\n\n    def predict(\n        self,\n        image: Any,\n        prompt: str = DEFAULT_PROMPT,\n        max_new_tokens: int = 8192,\n        do_sample: bool = False,\n        **generation_kwargs: Any,\n    ) -> Dict[str, Any]:\n        if self.model is None:\n            raise RuntimeError(\"模型未加载：请先调用 adapter.load()\")\n        import torch\n\n        messages = self.build_messages(prompt)\n        prompt_text = self.processor.apply_chat_template(\n            messages, add_generation_prompt=True\n        )\n        inputs = self.processor(\n            text=prompt_text, images=[image], return_tensors=\"pt\"\n        ).to(self.device)\n\n        t0 = time.perf_counter()\n        with torch.inference_mode():\n            generated_ids = self.model.generate(\n                **inputs,\n                max_new_tokens=max_new_tokens,\n                do_sample=do_sample,\n                **generation_kwargs,\n            )\n        latency = time.perf_counter() - t0\n\n        prompt_length = inputs.input_ids.shape[1]\n        doctags = self.processor.batch_decode(\n            generated_ids[:, prompt_length:],\n            skip_special_tokens=False,\n        )[0].lstrip()\n\n        return {\n            \"doctags\": doctags,\n            \"prompt\": prompt,\n            \"latency_sec\": round(latency, 3),\n            \"generation_config\": {\n                \"max_new_tokens\": max_new_tokens,\n                \"do_sample\": do_sample,\n            },\n            \"model_id\": self.model_id,\n            \"model_revision\": self.revision,\n            \"device\": self.device,\n            \"dtype\": self.dtype_name,\n        }\n\n\ndef doctags_to_docling(\n    doctags: str, image: Any, document_name: str = \"prediction\"\n) -> Tuple[Any, str]:\n    \"\"\"DocTags → DoclingDocument，兼容 docling-core 新旧 API（风险 R2）。\n\n    依次尝试官方已知的 API 形态，返回 (docling_document, api_path)。\n    api_path 会写入实验元数据，保证可追溯。\n    \"\"\"\n    errors: List[str] = []\n\n    def _try(label: str, fn: Any) -> Tuple[Any, str]:\n        try:\n            return fn(), label\n        except Exception as exc:  # noqa: BLE001\n            errors.append(f\"{label}: {exc}\")\n            return None, \"\"\n\n    try:\n        from docling_core.types.doc import DoclingDocument\n    except ImportError as exc:\n        raise ImportError(\n            \"缺少 docling-core。请在 Kaggle 上执行 \"\n            \"`!pip install docling-core`（见 requirements-kaggle.txt）。\"\n        ) from exc\n\n    # 路径 A：docling-core 2.x 新版（DocTagsDocument 已弃用）。\n    if hasattr(DoclingDocument, \"from_doctags_and_image_pairs\"):\n        for kwargs in (\n            {\"doctags_and_images\": [(doctags, image)]},\n            {\"doctags\": [doctags], \"images\": [image]},\n            {\"doctags\": doctags, \"images\": [image]},\n        ):\n            doc, label = _try(\n                f\"A:{list(kwargs)}\",\n                lambda kwargs=kwargs: DoclingDocument.from_doctags_and_image_pairs(\n                    **kwargs\n                ),\n            )\n            if doc is not None:\n                return doc, f\"DoclingDocument.from_doctags_and_image_pairs({list(kwargs)})\"\n\n    # 路径 B：模型卡旧 API（DocTagsDocument + DoclingDocument.load_from_doctags）。\n    try:\n        from docling_core.types.doc.document import DocTagsDocument\n\n        tags_doc, label = _try(\n            \"B:DocTagsDocument\",\n            lambda: DocTagsDocument.from_doctags_and_image_pairs([doctags], [image]),\n        )\n        if tags_doc is not None:\n            doc, label2 = _try(\n                \"B:load_from_doctags\",\n                lambda: DoclingDocument.load_from_doctags(\n                    tags_doc, document_name=document_name\n                ),\n            )\n            if doc is not None:\n                return doc, f\"DocTagsDocument.from_doctags_and_image_pairs → DoclingDocument.load_from_doctags\"\n    except ImportError:\n        pass\n\n    raise RuntimeError(\n        \"无法将 DocTags 转换为 DoclingDocument。已尝试：\\n- \"\n        + \"\\n- \".join(errors)\n        + \"\\n请按当前 docling-core 版本的官方文档核对 API（设计文档风险 R2）。\"\n    )\n\n\ndef doctags_to_markdown(doctags: str, image: Any) -> Tuple[str, str]:\n    \"\"\"便捷封装：DocTags → Markdown 文本。返回 (markdown, api_path)。\"\"\"\n    doc, api_path = doctags_to_docling(doctags, image)\n    return doc.export_to_markdown(), api_path\n\n\ndef model_summary(adapter: SmolDoclingAdapter) -> Dict[str, Any]:\n    \"\"\"模型与设备摘要（打印到 Notebook / 写入元数据）。\"\"\"\n    import torch\n\n    info: Dict[str, Any] = {\n        \"model_id\": adapter.model_id,\n        \"model_revision\": adapter.revision,\n        \"device\": adapter.device,\n        \"dtype\": adapter.dtype_name,\n        \"model_class\": getattr(adapter, \"model_class\", \"\"),\n        \"attn_implementation\": getattr(adapter, \"attn_implementation\", \"\"),\n        \"device_fallback_reason\": getattr(adapter, \"device_fallback_reason\", \"\"),\n    }\n    if adapter.model is not None:\n        total = sum(p.numel() for p in adapter.model.parameters())\n        info[\"total_parameters\"] = total\n    if adapter.device == \"cuda\":\n        props = torch.cuda.get_device_properties(0)\n        info[\"gpu\"] = props.name\n        info[\"vram_gb\"] = round(props.total_memory / 1024**3, 1)\n        info[\"memory_allocated_gb\"] = round(torch.cuda.memory_allocated() / 1024**3, 2)\n    return info\n", "src/prompts.py": "\"\"\"Prompt 版本管理。\n\nPrompt 本身就是实验变量（任务文件 §八）。Notebook 03（Phase 3）会系统比较\nV0–V3；Phase 2 的 Baseline 使用官方默认 V0。每次推理必须记录 prompt_id。\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Dict\n\nPROMPT_V0 = \"Convert this page to docling.\"\n\nPROMPT_V1 = (\n    \"Convert this page to docling. \"\n    \"Extract all text content as accurately as possible.\"\n)\n\nPROMPT_V2 = (\n    \"Convert this page to docling. \"\n    \"Preserve the reading order, table structure, formulas and layout hierarchy.\"\n)\n\nPROMPT_V3 = (\n    \"You are a document parsing engine. Convert this page to the DocTags \"\n    \"structured format. Capture every text block, title, table (with full \"\n    \"structure), formula (as LaTeX), figure and caption. Keep the correct \"\n    \"reading order and include bounding boxes for every element.\"\n)\n\nPROMPTS: Dict[str, Dict[str, object]] = {\n    \"v0\": {\n        \"id\": \"v0\",\n        \"text\": PROMPT_V0,\n        \"focus\": \"官方默认 full conversion\",\n        \"official\": True,\n    },\n    \"v1\": {\n        \"id\": \"v1\",\n        \"text\": PROMPT_V1,\n        \"focus\": \"强调 OCR 准确率\",\n        \"official\": False,\n    },\n    \"v2\": {\n        \"id\": \"v2\",\n        \"text\": PROMPT_V2,\n        \"focus\": \"强调阅读顺序/表格/公式/版面\",\n        \"official\": False,\n    },\n    \"v3\": {\n        \"id\": \"v3\",\n        \"text\": PROMPT_V3,\n        \"focus\": \"面向结构化 Document Parsing\",\n        \"official\": False,\n    },\n}\n\n\ndef get_prompt(prompt_id: str) -> str:\n    if prompt_id not in PROMPTS:\n        raise ValueError(f\"未知 prompt_id: {prompt_id}，可选：{sorted(PROMPTS)}\")\n    return str(PROMPTS[prompt_id][\"text\"])\n", "src/inference.py": "\"\"\"批量推理、缓存与断点恢复。\n\n设计原则（任务文件 §九、docs/notebook-design.md §8.5/8.6）：\n- 三种模式：fast / teaching / research（页数在 configs/default.yaml 定义）；\n- 每个样本写入独立 JSON 缓存，重启后自动跳过已有输出；\n- 记录 prompt_id、model revision、image_id、generation_config、latency；\n- 产物写入 results/baseline/。\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Sequence\n\nfrom . import data\nfrom .model import DocumentModelAdapter\nfrom .prompts import get_prompt\n\n\ndef environment_snapshot() -> Dict[str, Any]:\n    \"\"\"GPU / CUDA / PyTorch / transformers / docling 版本快照。\n\n    任务文件 §三要求每个 Notebook 自动输出这些信息；\n    结果同时写入 experiment_metadata.json。\n    \"\"\"\n    import platform\n\n    import torch\n\n    snap: Dict[str, Any] = {\n        \"python\": sys.version.split()[0],\n        \"platform\": platform.platform(),\n        \"torch\": torch.__version__,\n        \"cuda_available\": torch.cuda.is_available(),\n        \"gpu\": [],\n    }\n    if torch.cuda.is_available():\n        snap[\"cuda_version\"] = torch.version.cuda\n        snap[\"gpu\"] = [\n            {\n                \"name\": torch.cuda.get_device_name(i),\n                \"vram_gb\": round(\n                    torch.cuda.get_device_properties(i).total_memory / 1024**3, 1\n                ),\n            }\n            for i in range(torch.cuda.device_count())\n        ]\n    for lib in (\"transformers\", \"docling_core\", \"pillow\", \"numpy\", \"pandas\"):\n        try:\n            mod = __import__(lib)\n            snap[lib] = getattr(mod, \"__version__\", \"unknown\")\n        except ImportError:\n            snap[lib] = None\n    try:\n        out = subprocess.run(\n            [\"nvidia-smi\", \"--query-gpu=name,memory.total\", \"--format=csv,noheader\"],\n            capture_output=True,\n            text=True,\n            timeout=30,\n        )\n        if out.returncode == 0:\n            snap[\"nvidia_smi\"] = out.stdout.strip()\n    except Exception:  # noqa: BLE001\n        snap[\"nvidia_smi\"] = None\n    return snap\n\n\ndef run_baseline(\n    annotations: Sequence[Dict[str, Any]],\n    data_root: Path,\n    adapter: DocumentModelAdapter,\n    output_dir: Path,\n    mode: str = \"fast\",\n    config: Optional[Dict[str, Any]] = None,\n    seed: int = 42,\n    prompt_id: str = \"v0\",\n    skip_existing: bool = True,\n    n_pages: Optional[int] = None,\n    predict_kwargs: Optional[Dict[str, Any]] = None,\n) -> Path:\n    \"\"\"运行 Baseline 批量推理。\n\n    返回 manifest 路径：<output_dir>/manifest.jsonl\n    \"\"\"\n    cfg = config or {}\n    if n_pages is None:\n        n_pages = int(cfg.get(\"modes\", {}).get(mode, {\"fast\": 12}.get(mode, 12)))\n    out = Path(output_dir)\n    pred_dir = out / \"predictions\"\n    doctags_dir = out / \"doctags\"\n    pred_dir.mkdir(parents=True, exist_ok=True)\n    doctags_dir.mkdir(parents=True, exist_ok=True)\n\n    try:\n        from tqdm.auto import tqdm\n    except ImportError:\n        def tqdm(it: Any, **_: Any) -> Any:  # type: ignore[misc]\n            return it\n\n    pages = data.select_pages(annotations, n=n_pages, seed=seed)\n    rows: List[Dict[str, Any]] = []\n    skipped = 0\n    t_start = time.perf_counter()\n    for page in tqdm(pages, desc=f\"baseline [{mode}]\"):\n        image_id = data.sample_id(page)\n        pred_path = pred_dir / f\"{image_id}.json\"\n        if skip_existing and pred_path.is_file():\n            try:\n                rows.append(data.read_json(pred_path))\n            except Exception:  # noqa: BLE001\n                pred_path.unlink(missing_ok=True)\n            else:\n                skipped += 1\n                continue\n        image = data.load_page_image(data_root, page)\n        prediction = adapter.predict(\n            image, prompt=get_prompt(prompt_id), **(predict_kwargs or {})\n        )\n        prediction[\"image_id\"] = image_id\n        prediction[\"prompt_id\"] = prompt_id\n        prediction[\"document_type\"] = data.page_attribute(page).get(\n            \"data_source\", \"unknown\"\n        )\n        prediction[\"language\"] = data.page_attribute(page).get(\"language\", \"unknown\")\n        prediction[\"layout\"] = data.page_attribute(page).get(\"layout\", \"unknown\")\n        prediction[\"subset\"] = data.subset_of(page)\n        adapter.save_prediction(prediction, pred_path)\n        (doctags_dir / f\"{image_id}.dt\").write_text(\n            prediction[\"doctags\"], encoding=\"utf-8\"\n        )\n        rows.append(prediction)\n\n    manifest_path = out / \"manifest.jsonl\"\n    with open(manifest_path, \"w\", encoding=\"utf-8\") as f:\n        for row in rows:\n            f.write(json.dumps(row, ensure_ascii=False) + \"\\n\")\n\n    latencies = [float(r[\"latency_sec\"]) for r in rows if \"latency_sec\" in r]\n    summary = {\n        \"mode\": mode,\n        \"requested_pages\": n_pages,\n        \"completed\": len(rows),\n        \"skipped_from_cache\": skipped,\n        \"prompt_id\": prompt_id,\n        \"seed\": seed,\n        \"total_latency_sec\": round(sum(latencies), 2),\n        \"mean_latency_sec\": round(sum(latencies) / len(latencies), 3) if latencies else None,\n        \"wall_sec\": round(time.perf_counter() - t_start, 2),\n        \"model_id\": rows[0][\"model_id\"] if rows else None,\n        \"model_revision\": rows[0][\"model_revision\"] if rows else None,\n    }\n    data.write_json(summary, out / \"summary.json\")\n    return manifest_path\n", "src/evaluation.py": "\"\"\"OmniDocBench 官方评测封装。\n\n红线（任务文件 §二十、docs/notebook-design.md §9）：\n- 只调用官方 Evaluation，不自行发明替代指标；\n- 官方评测仓库无 release/tag，锁定 commit 193627ae…；\n- 本模块的 normalized_edit_distance 仅为 smoke 自检，\n  明确标注「非官方指标」，不得用于论文/报告结论。\n\n⚠️ 官方 CLI 入口与 prediction 格式细节需在首次 Kaggle 运行时按锁定 commit\n的 README / configs 核对（设计文档第 11 节待核实清单）。\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport subprocess\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Sequence\n\nfrom . import data\n\nOFFICIAL_REPO_URL = \"https://github.com/opendatalab/OmniDocBench.git\"\nOFFICIAL_COMMIT = \"193627ae9e97d89188468ed1ee3b7a856ff76044\"\n\n\ndef ensure_eval_repo(base_dir: Path, commit: str = OFFICIAL_COMMIT) -> Path:\n    \"\"\"克隆或复用官方评测仓库并锁定 commit。\"\"\"\n    repo = Path(base_dir) / \"OmniDocBench\"\n    if repo.is_dir():\n        head = subprocess.run(\n            [\"git\", \"rev-parse\", \"HEAD\"],\n            cwd=repo,\n            capture_output=True,\n            text=True,\n        )\n        if head.returncode == 0 and head.stdout.strip().startswith(commit):\n            return repo\n        raise RuntimeError(\n            f\"已有评测仓库 {repo}，但 commit 与锁定值 {commit} 不一致。\"\n            \"请手动删除后重试（git clone 在 Kaggle 上约 1-2 分钟）。\"\n        )\n    repo.parent.mkdir(parents=True, exist_ok=True)\n    subprocess.run([\"git\", \"clone\", OFFICIAL_REPO_URL, str(repo)], check=True)\n    subprocess.run([\"git\", \"checkout\", commit], cwd=repo, check=True)\n    return repo\n\n\ndef export_markdown_predictions(\n    predictions_dir: Path,\n    data_root: Path,\n    output_dir: Path,\n) -> List[Path]:\n    \"\"\"把 baseline 缓存的 doctags 转成 Markdown 文件（md2md 评测输入）。\n\n    输入：results/baseline/predictions/*.json（含 doctags 字段）\n    输出：<output_dir>/<image_id>.md\n    \"\"\"\n    from .model import doctags_to_markdown\n\n    pred_dir = Path(predictions_dir)\n    out = Path(output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    written: List[Path] = []\n    for pred_file in sorted(pred_dir.glob(\"*.json\")):\n        pred = data.read_json(pred_file)\n        image_id = pred[\"image_id\"]\n        image_path = Path(data_root) / data.IMAGES_DIRNAME / f\"{image_id}.jpg\"\n        if not image_path.is_file():\n            image_path = Path(data_root) / data.IMAGES_DIRNAME / f\"{image_id}.png\"\n        if not image_path.is_file():\n            raise FileNotFoundError(f\"找不到页面图像：{image_id}\")\n        from PIL import Image\n\n        with Image.open(image_path) as img:\n            image = img.convert(\"RGB\")\n        markdown, _api = doctags_to_markdown(pred[\"doctags\"], image)\n        out_file = out / f\"{image_id}.md\"\n        out_file.write_text(markdown, encoding=\"utf-8\")\n        written.append(out_file)\n    return written\n\n\n# Docling item label → OmniDocBench category_type 的候选映射。\n# ⚠️ 首次运行时按官方 README 的 category 定义核对并修正。\nCATEGORY_MAP = {\n    \"title\": \"title\",\n    \"text\": \"text_block\",\n    \"table\": \"table\",\n    \"picture\": \"figure\",\n    \"formula\": \"equation_isolated\",\n    \"section_header\": \"title\",\n    \"list_item\": \"text_block\",\n    \"code\": \"code_txt\",\n    \"caption\": \"table_caption\",\n}\n\n\ndef export_end2end_predictions(\n    predictions_dir: Path,\n    data_root: Path,\n    output_dir: Path,\n) -> Path:\n    \"\"\"导出 end2end 评测候选格式（layout_dets 镜像官方 GT 结构）。\n\n    ⚠️ 候选格式：字段与类别映射需按锁定 commit 的官方 README 核对。\n    本函数会在导出时打印警告，并在 metadata 中记录 mapping 版本。\n    \"\"\"\n    from .model import doctags_to_docling\n\n    pred_dir = Path(predictions_dir)\n    out = Path(output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    preds: List[Dict[str, Any]] = []\n    for pred_file in sorted(pred_dir.glob(\"*.json\")):\n        pred = data.read_json(pred_file)\n        image_id = pred[\"image_id\"]\n        image_path = Path(data_root) / data.IMAGES_DIRNAME / f\"{image_id}.jpg\"\n        if not image_path.is_file():\n            image_path = Path(data_root) / data.IMAGES_DIRNAME / f\"{image_id}.png\"\n        from PIL import Image\n\n        with Image.open(image_path) as img:\n            image = img.convert(\"RGB\")\n        doc, _api = doctags_to_docling(pred[\"doctags\"], image)\n        layout_dets: List[Dict[str, Any]] = []\n        for order, (item, _level) in enumerate(doc.iterate_items()):\n            bbox = None\n            if item.prov:\n                bbox = item.prov[0].bbox\n            category = CATEGORY_MAP.get(str(item.label), \"abandon\")\n            det: Dict[str, Any] = {\n                \"category_type\": category,\n                \"order\": order,\n                \"ignore\": False,\n            }\n            if bbox is not None:\n                det[\"poly\"] = [\n                    bbox.l,\n                    bbox.t,\n                    bbox.r,\n                    bbox.t,\n                    bbox.r,\n                    bbox.b,\n                    bbox.l,\n                    bbox.b,\n                ]\n            text = getattr(item, \"text\", None)\n            if text:\n                det[\"text\"] = text\n            layout_dets.append(det)\n        preds.append({\"layout_dets\": layout_dets, \"page_info\": {\"image_path\": image_id}})\n    out_file = out / \"pred_end2end.json\"\n    data.write_json(preds, out_file)\n    return out_file\n\n\ndef run_official_eval(\n    config: Dict[str, Any],\n    repo: Path,\n    gt_json: Path,\n    pred_dir: Path,\n    output_dir: Path,\n    include_cdm: bool = False,\n) -> Optional[Path]:\n    \"\"\"调用官方评测入口 pdf_validation.py（锁定 commit 核实于 2026-08-13）。\n\n    先按官方 configs/end2end.yaml 结构生成评测配置（CDM 默认关闭），\n    再执行 `python pdf_validation.py --config <yaml>`。\n    模板为空时返回 None（不做假跑）。\n    \"\"\"\n    section = config.get(\"omnidocbench_eval\", {})\n    template = section.get(\"end2end_cmd_template\", \"\") or \"\"\n    if not template:\n        return None\n    cfg_path = write_end2end_config(\n        repo, gt_json, pred_dir, output_dir, include_cdm=include_cdm\n    )\n    cmd = (\n        template.replace(\"{repo}\", str(repo))\n        .replace(\"{config}\", str(cfg_path))\n    )\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    log = output_dir / \"official_end2end.log\"\n    with open(log, \"w\", encoding=\"utf-8\") as f:\n        proc = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)\n    if proc.returncode != 0:\n        raise RuntimeError(f\"官方评测失败（returncode={proc.returncode}），见 {log}\")\n    return log\n\n\ndef write_end2end_config(\n    repo: Path,\n    gt_json: Path,\n    pred_dir: Path,\n    output_dir: Path,\n    include_cdm: bool = False,\n) -> Path:\n    \"\"\"按官方 configs/end2end.yaml（锁定 commit）结构生成评测配置。\n\n    - text_block: Edit_dist；display_formula: Edit_dist（+CDM 可选）；\n    - table: TEDS + Edit_dist；reading_order: Edit_dist；\n    - match_method: quick_match（官方推荐）。\n    \"\"\"\n    cdm_lines = \"\\n      - CDM\" if include_cdm else \"      # CDM 需要独立环境，默认关闭\"\n    content = (\n        \"end2end_eval:\\n\"\n        \"  metrics:\\n\"\n        \"    text_block:\\n\"\n        \"      metric:\\n\"\n        \"      - Edit_dist\\n\"\n        \"    display_formula:\\n\"\n        \"      metric:\\n\"\n        \"      - Edit_dist\\n\"\n        f\"{cdm_lines}\\n\"\n        \"    table:\\n\"\n        \"      metric:\\n\"\n        \"      - TEDS\\n\"\n        \"      - Edit_dist\\n\"\n        \"    reading_order:\\n\"\n        \"      metric:\\n\"\n        \"      - Edit_dist\\n\"\n        \"  dataset:\\n\"\n        \"    dataset_name: end2end_dataset\\n\"\n        \"    ground_truth:\\n\"\n        f\"      data_path: {Path(gt_json).resolve()}\\n\"\n        \"    prediction:\\n\"\n        f\"      data_path: {Path(pred_dir).resolve()}\\n\"\n        \"    match_method: quick_match\\n\"\n    )\n    out = Path(output_dir) / \"end2end_custom.yaml\"\n    out.parent.mkdir(parents=True, exist_ok=True)\n    out.write_text(content, encoding=\"utf-8\")\n    return out\n\n\ndef prepare_gt_subset(\n    pages: Sequence[Dict[str, Any]], out_path: Path\n) -> Path:\n    \"\"\"把若干官方页面写成评测 GT 子集 JSON（end2end 输入）。\"\"\"\n    return data.write_json(list(pages), out_path)\n\n\ndef gt_subset_for_predictions(\n    pred_dir: Path,\n    annotations: Sequence[Dict[str, Any]],\n    out_path: Path,\n) -> Path:\n    \"\"\"按 predictions 目录中的 image_id 抽取对应 GT 页面。\"\"\"\n    ids = {p.stem for p in Path(pred_dir).glob(\"*.md\")}\n    pages = [p for p in annotations if data.sample_id(p) in ids]\n    if not pages:\n        raise ValueError(\"predictions 目录中没有与 GT 匹配的 .md 文件\")\n    return prepare_gt_subset(pages, out_path)\n\n\ndef normalized_edit_distance(pred_text: str, gt_text: str) -> float:\n    \"\"\"⚠️ 非官方 smoke 自检指标：归一化编辑距离相似度（0-1，越大越好）。\n\n    只用于验证 pipeline 是否贯通，不代表 OmniDocBench 官方成绩。\n    \"\"\"\n    if not gt_text and not pred_text:\n        return 1.0\n    if not gt_text or not pred_text:\n        return 0.0\n    m, n = len(pred_text), len(gt_text)\n    dp = list(range(n + 1))\n    for i in range(1, m + 1):\n        prev = dp[0]\n        dp[0] = i\n        for j in range(1, n + 1):\n            tmp = dp[j]\n            cost = 0 if pred_text[i - 1] == gt_text[j - 1] else 1\n            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + cost)\n            prev = tmp\n    return 1.0 - dp[n] / max(m, n)\n\n\ndef sanity_check(\n    predictions_dir: Path,\n    annotations: Sequence[Dict[str, Any]],\n    data_root: Path,\n) -> List[Dict[str, Any]]:\n    \"\"\"⚠️ 非官方 smoke 自检：预测 Markdown vs GT 文本块的归一化编辑距离。\"\"\"\n    from .model import doctags_to_markdown\n    from PIL import Image\n\n    rows: List[Dict[str, Any]] = []\n    for pred_file in sorted(Path(predictions_dir).glob(\"*.json\")):\n        pred = data.read_json(pred_file)\n        image_id = pred[\"image_id\"]\n        gt_page = next(\n            (p for p in annotations if data.sample_id(p) == image_id), None\n        )\n        if gt_page is None:\n            continue\n        image_path = Path(data_root) / data.IMAGES_DIRNAME / f\"{image_id}.jpg\"\n        if not image_path.is_file():\n            image_path = Path(data_root) / data.IMAGES_DIRNAME / f\"{image_id}.png\"\n        with Image.open(image_path) as img:\n            image = img.convert(\"RGB\")\n        markdown, _api = doctags_to_markdown(pred[\"doctags\"], image)\n        gt_text = \"\\n\".join(\n            d.get(\"text\", \"\")\n            for d in gt_page.get(\"layout_dets\", []) or []\n            if d.get(\"text\")\n        )\n        rows.append(\n            {\n                \"image_id\": image_id,\n                \"document_type\": pred.get(\"document_type\"),\n                \"language\": pred.get(\"language\"),\n                \"layout\": pred.get(\"layout\"),\n                \"sanity_ned\": round(normalized_edit_distance(markdown, gt_text), 4),\n                \"metric_kind\": \"non_official_smoke_only\",\n            }\n        )\n    return rows\n\n\ndef build_summary_table(\n    rows: Sequence[Dict[str, Any]],\n    group_keys: Sequence[str] = (\"document_type\", \"language\", \"layout\"),\n    score_key: str = \"sanity_ned\",\n) -> Dict[str, Any]:\n    \"\"\"总体 + 分组汇总（均值与样本数）。\"\"\"\n    import collections\n\n    def agg(items: Sequence[Dict[str, Any]]) -> Dict[str, Any]:\n        scores = [float(r[score_key]) for r in items if score_key in r]\n        return {\n            \"n\": len(items),\n            f\"mean_{score_key}\": round(sum(scores) / len(scores), 4) if scores else None,\n        }\n\n    summary: Dict[str, Any] = {\"overall\": agg(list(rows))}\n    for key in group_keys:\n        groups: Dict[str, List[Dict[str, Any]]] = collections.defaultdict(list)\n        for r in rows:\n            groups[str(r.get(key, \"unknown\"))].append(r)\n        summary[key] = {k: agg(v) for k, v in sorted(groups.items())}\n    return summary\n", "src/visualization.py": "\"\"\"标注与输出可视化（Notebook 01/08 共用）。\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional\n\nCATEGORY_COLORS: Dict[str, str] = {\n    \"title\": \"#1f77b4\",\n    \"text_block\": \"#2ca02c\",\n    \"list_group\": \"#98df8a\",\n    \"reference\": \"#c5b0d5\",\n    \"figure\": \"#ff7f0e\",\n    \"figure_caption\": \"#ffbb78\",\n    \"table\": \"#d62728\",\n    \"table_caption\": \"#ff9896\",\n    \"equation_isolated\": \"#9467bd\",\n    \"equation_caption\": \"#c49c94\",\n    \"header\": \"#8c564b\",\n    \"footer\": \"#8c564b\",\n    \"page_number\": \"#e377c2\",\n    \"abandon\": \"#d9d9d9\",\n}\nDEFAULT_COLOR = \"#7f7f7f\"\n\n\ndef draw_annotations(\n    image: Any,\n    page: Dict[str, Any],\n    max_regions: int = 40,\n    title: Optional[str] = None,\n) -> Any:\n    \"\"\"绘制页面标注：按 category 着色 + 阅读顺序编号。返回 matplotlib Figure。\"\"\"\n    import matplotlib.pyplot as plt\n    from matplotlib.patches import Polygon\n\n    fig, ax = plt.subplots(figsize=(12, 12))\n    ax.imshow(image)\n    ax.set_axis_off()\n\n    layout_dets = [d for d in page.get(\"layout_dets\", []) or [] if not d.get(\"ignore\")]\n    for det in layout_dets[:max_regions]:\n        poly = det.get(\"poly\")\n        if not poly or len(poly) < 8:\n            continue\n        points = [(poly[i], poly[i + 1]) for i in range(0, 8, 2)]\n        cat = det.get(\"category_type\", \"unknown\")\n        color = CATEGORY_COLORS.get(cat, DEFAULT_COLOR)\n        ax.add_patch(\n            Polygon(points, closed=True, fill=False, edgecolor=color, linewidth=1.5)\n        )\n        order = det.get(\"order\")\n        if order is not None:\n            ax.text(points[0][0] + 2, points[0][1] - 4, str(order), fontsize=7, color=color)\n    if title:\n        ax.set_title(title, fontsize=12)\n    handles = [\n        plt.Line2D([0], [0], color=color, lw=2, label=cat)\n        for cat, color in CATEGORY_COLORS.items()\n    ]\n    ax.legend(handles=handles, loc=\"lower left\", bbox_to_anchor=(1.01, 0), fontsize=8)\n    plt.tight_layout()\n    return fig\n\n\ndef save_figure(fig: Any, path: Path) -> Path:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(path, dpi=110, bbox_inches=\"tight\")\n    return path\n", "src/ablation.py": "\"\"\"消融实验记录与可视化（Phase 4，Notebook 09）。\n\n红线：一次只改变一个主要变量；每个实验必须记录固定量与变量\n（模型 revision、数据划分、prompt、评测 commit、唯一变量）。\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Sequence\n\nfrom . import data\n\n\ndef ablation_record(\n    ablation: str,\n    variable: str,\n    value: str,\n    metrics: Dict[str, Any],\n    fixed: Dict[str, Any],\n) -> Dict[str, Any]:\n    \"\"\"构造一行消融记录（写入 ablation_results.csv）。\"\"\"\n    record = {\n        \"ablation\": ablation,\n        \"variable\": variable,\n        \"value\": value,\n        **metrics,\n        \"fixed\": fixed,\n    }\n    return record\n\n\ndef write_ablation_csv(rows: Sequence[Dict[str, Any]], path: Path) -> Path:\n    import csv\n\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    keys: List[str] = []\n    for row in rows:\n        for k in row:\n            if k not in keys:\n                keys.append(k)\n    with open(path, \"w\", newline=\"\", encoding=\"utf-8\") as f:\n        writer = csv.DictWriter(f, fieldnames=keys)\n        writer.writeheader()\n        for row in rows:\n            writer.writerow({k: row.get(k, \"\") for k in keys})\n    return path\n\n\ndef plot_ablation(rows: Sequence[Dict[str, Any]], x_key: str, y_keys: Sequence[str], title: str):\n    \"\"\"折线图：x 为变量取值，y 为若干指标。返回 matplotlib Figure。\"\"\"\n    import matplotlib.pyplot as plt\n\n    fig, ax = plt.subplots(figsize=(8, 5))\n    for y_key in y_keys:\n        xs = [r[\"value\"] for r in rows]\n        ys = [r.get(y_key) for r in rows]\n        ax.plot(xs, ys, marker=\"o\", label=y_key)\n    ax.set_xlabel(x_key)\n    ax.set_ylabel(\"metric\")\n    ax.set_title(title)\n    ax.legend()\n    ax.grid(True, alpha=0.3)\n    plt.tight_layout()\n    return fig\n", "src/error_analysis.py": "\"\"\"错误分类学与案例选择（Phase 4，Notebook 08 核心）。\n\n⚠️ 教学定位：本模块的自动分类是「启发式」，用于给学生提供可复查的\n错误案例种子与统计入口，**不是官方评测结论**。结论必须回到\n官方指标（Notebook 07）与人工复核。\n\"\"\"\n\nfrom __future__ import annotations\n\nimport collections\nimport difflib\nimport json\nimport re\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Sequence\n\nfrom . import data\nfrom .evaluation import normalized_edit_distance\n\nERROR_TAXONOMY = [\n    \"ocr_error\",\n    \"layout_error\",\n    \"reading_order_error\",\n    \"table_error\",\n    \"formula_error\",\n    \"missing_content\",\n    \"hallucination\",\n    \"repetition\",\n    \"structure_error\",\n]\n\n\ndef normalize_text(text: str) -> str:\n    \"\"\"小写 + 压缩空白，用于文本对齐。\"\"\"\n    return re.sub(r\"\\s+\", \" \", (text or \"\").lower()).strip()\n\n\ndef _diff_ratios(pred: str, gt: str) -> Dict[str, float]:\n    \"\"\"用 SequenceMatcher 估计缺失与幻觉内容占比（启发式）。\"\"\"\n    pred_n = normalize_text(pred)\n    gt_n = normalize_text(gt)\n    if not gt_n:\n        return {\"missing_ratio\": 0.0, \"hallucination_ratio\": 0.0, \"similarity\": 0.0}\n    if not pred_n:\n        return {\"missing_ratio\": 1.0, \"hallucination_ratio\": 0.0, \"similarity\": 0.0}\n    matcher = difflib.SequenceMatcher(None, gt_n, pred_n)\n    missing = 0\n    hallucinated = 0\n    for tag, i1, i2, j1, j2 in matcher.get_opcodes():\n        if tag in (\"delete\", \"replace\"):\n            missing += i2 - i1\n        if tag in (\"insert\", \"replace\"):\n            hallucinated += j2 - j1\n    return {\n        \"missing_ratio\": round(missing / max(len(gt_n), 1), 4),\n        \"hallucination_ratio\": round(hallucinated / max(len(pred_n), 1), 4),\n        \"similarity\": round(matcher.ratio(), 4),\n    }\n\n\ndef _has_repetition(text: str, min_len: int = 24) -> bool:\n    \"\"\"检测明显的重复片段（启发式）。\"\"\"\n    return bool(re.search(r\"(.{%d,}?)\\1\" % min_len, normalize_text(text)))\n\n\ndef classify_case(\n    prediction: Dict[str, Any],\n    page: Dict[str, Any],\n    pred_markdown: str,\n) -> Dict[str, Any]:\n    \"\"\"把一条预测 + 对应 GT 页面分类为错误类型集合（启发式）。\n\n    返回 error_cases.json 的单条记录。\n    \"\"\"\n    attr = data.page_attribute(page)\n    gt_text = \"\\n\".join(\n        d.get(\"text\", \"\")\n        for d in page.get(\"layout_dets\", []) or []\n        if d.get(\"text\")\n    )\n    ratios = _diff_ratios(pred_markdown, gt_text)\n    ned = normalized_edit_distance(normalize_text(pred_markdown), normalize_text(gt_text))\n\n    block_cats = {\n        d.get(\"category_type\") for d in page.get(\"layout_dets\", []) or []\n    }\n    doctags = prediction.get(\"doctags\", \"\")\n    error_types: List[str] = []\n    if ned < 0.65:\n        error_types.append(\"ocr_error\")\n    if ratios[\"missing_ratio\"] > 0.35:\n        error_types.append(\"missing_content\")\n    if ratios[\"hallucination_ratio\"] > 0.25:\n        error_types.append(\"hallucination\")\n    if _has_repetition(pred_markdown):\n        error_types.append(\"repetition\")\n    if \"table\" in block_cats and \"<table>\" not in doctags:\n        error_types.append(\"table_error\")\n    if (\n        {\"equation_isolated\", \"equation_caption\", \"equation_semantic\"} & block_cats\n        and \"<formula>\" not in doctags\n    ):\n        error_types.append(\"formula_error\")\n    if attr.get(\"layout\") in (\"double_column\", \"three_column\", \"1andmore_column\"):\n        if ratios[\"missing_ratio\"] > 0.2 and \"missing_content\" in error_types:\n            error_types.append(\"reading_order_error\")\n    if \"<table>\" in doctags and doctags.count(\"<table>\") < sum(\n        1\n        for d in page.get(\"layout_dets\", []) or []\n        if d.get(\"category_type\") == \"table\"\n    ):\n        error_types.append(\"structure_error\")\n\n    return {\n        \"image_id\": prediction.get(\"image_id\", data.sample_id(page)),\n        \"document_type\": attr.get(\"data_source\", \"unknown\"),\n        \"language\": attr.get(\"language\", \"unknown\"),\n        \"layout\": attr.get(\"layout\", \"unknown\"),\n        \"subset\": data.subset_of(page),\n        \"error_types\": sorted(set(error_types)),\n        \"sanity_ned\": round(ned, 4),\n        \"gt_chars\": len(gt_text),\n        \"pred_chars\": len(pred_markdown),\n        \"evidence\": f\"results/baseline/predictions/{prediction.get('image_id')}.json\",\n        \"notes\": \"\",\n    }\n\n\ndef build_error_cases(\n    predictions_dir: Path,\n    annotations: Sequence[Dict[str, Any]],\n    data_root: Path,\n    output_path: Optional[Path] = None,\n) -> List[Dict[str, Any]]:\n    \"\"\"对 predictions_dir 中所有预测生成错误案例记录，写入 error_cases.json。\"\"\"\n    from .model import doctags_to_markdown\n    from PIL import Image\n\n    cases: List[Dict[str, Any]] = []\n    ann_by_id = {data.sample_id(p): p for p in annotations}\n    for pred_file in sorted(Path(predictions_dir).glob(\"*.json\")):\n        prediction = data.read_json(pred_file)\n        image_id = prediction.get(\"image_id\")\n        page = ann_by_id.get(image_id)\n        if page is None:\n            continue\n        image_path = Path(data_root) / data.IMAGES_DIRNAME / f\"{image_id}.jpg\"\n        if not image_path.is_file():\n            image_path = Path(data_root) / data.IMAGES_DIRNAME / f\"{image_id}.png\"\n        with Image.open(image_path) as img:\n            image = img.convert(\"RGB\")\n        markdown, _api = doctags_to_markdown(prediction[\"doctags\"], image)\n        cases.append(classify_case(prediction, page, markdown))\n    if output_path is not None:\n        data.write_json(cases, output_path)\n    return cases\n\n\ndef taxonomy_summary(cases: Sequence[Dict[str, Any]]) -> Dict[str, int]:\n    counter: Dict[str, int] = collections.Counter()\n    for case in cases:\n        for t in case[\"error_types\"]:\n            counter[t] += 1\n    return dict(counter)\n\n\ndef select_worst_cases(cases: Sequence[Dict[str, Any]], n: int = 20) -> List[Dict[str, Any]]:\n    \"\"\"Top-N 最差页面（按 sanity_ned 升序）。\"\"\"\n    return sorted(cases, key=lambda c: c[\"sanity_ned\"])[:n]\n\n\ndef select_improvement_cases(\n    baseline_cases: Sequence[Dict[str, Any]],\n    finetuned_cases: Sequence[Dict[str, Any]],\n    n: int = 10,\n) -> Dict[str, List[Dict[str, Any]]]:\n    \"\"\"训练前后对比：改善最大 / 退化（regression）的页面。\"\"\"\n    ft = {c[\"image_id\"]: c for c in finetuned_cases}\n    diffs = []\n    for base in baseline_cases:\n        other = ft.get(base[\"image_id\"])\n        if other is None:\n            continue\n        diffs.append(\n            (other[\"sanity_ned\"] - base[\"sanity_ned\"], base[\"image_id\"], base, other)\n        )\n    diffs.sort(key=lambda x: x[0])\n    regressions = [d[3] for d in diffs[:n]]\n    diffs.sort(key=lambda x: -x[0])\n    improvements = [d[3] for d in diffs[:n]]\n    return {\"improved\": improvements, \"regressed\": regressions}\n", "configs/default.yaml": "# 全局实验配置（Phase 2）。\n# 修改本文件后重跑 Notebook 即可；实验产物自动记录所用配置。\n#\n# Kaggle 首次实测（2026-08-13，smoke v6 全步骤通过）：\n# GPU Tesla P100-PCIE-16GB + 镜像 torch 2.10.0+cu128 无 sm_60 kernel，\n# src/model.py 自动回退 CPU 并记录 device_fallback_reason。\n# 实测：数据集下载 89.8s（images 1.38 GB）；模型加载 14.3s；\n# CPU 单页推理约 894s（max_new_tokens=8192）；fast 12 页 ≈ 3h。\n# teaching/research 模式在 CPU 上不可行，排障方案见 notebooks/README.md。\n\nproject:\n  seed: 42\n  description: \"SmolDocling x OmniDocBench teaching track\"\n\nmodel:\n  # 官方模型与锁定 revision（docs/notebook-design.md §3）\n  id: docling-project/SmolDocling-256M-preview\n  revision: ce51f56c4ebe36e0b1c3a55f67b261ba22a50bf8\n  dtype: auto        # auto | bfloat16 | float16 | float32\n  attention: auto    # auto | flash_attention_2 | eager\n\ndataset:\n  # 官方数据集与锁定 revision（docs/notebook-design.md §4）\n  id: opendatalab/OmniDocBench\n  revision: aa1ee96d106dbe53d0ae59474d75c6e6d9b53fec\n  annotation_file: OmniDocBench.json\n  images_dir: images\n\nomnidocbench_eval:\n  repo_url: https://github.com/opendatalab/OmniDocBench.git\n  # 官方仓库无 release/tag，锁定 main 最新 commit（2026-08-13 核实）\n  commit: 193627ae9e97d89188468ed1ee3b7a856ff76044\n  # 官方 CLI 入口（2026-08-13 从锁定 commit 的 README 与 configs/end2end.yaml 核实）：\n  #   python pdf_validation.py --config <yaml>\n  # - 该 commit 的 configs/ 只有 end2end.yaml（评测代码 v1.7）；\n  #   md2md.yaml 只存在于 v1_5 分支 → md2md 模板留空；\n  # - end2end 的 prediction 是「每页一个 .md 文件」的文件夹，\n  #   ground_truth 是 OmniDocBench 标注 JSON（可给页面子集）；\n  # - CDM 需要独立环境（Node/KaTeX），首跑配置由 src/evaluation.py 生成时默认关闭。\n  entry_script: pdf_validation.py\n  md2md_cmd_template: \"\"\n  end2end_cmd_template: \"python {repo}/pdf_validation.py --config {config}\"\n\nmodes:\n  # 三种运行模式的页数（任务文件 §九）\n  fast: 12\n  teaching: 100\n  research: 1651\n\ngeneration:\n  max_new_tokens: 8192\n  do_sample: false\n  prompt_id: v0        # Baseline 使用官方默认 prompt\n\npaths:\n  results_dir: results\n  baseline_dir: results/baseline\n  benchmark_dir: results/benchmark\n  metadata_file: results/experiment_metadata.json\n  manifest_file: results/experiment_manifest.json\n", "prompts/v0.txt": "# Prompt v0 — 官方默认 full conversion\nConvert this page to docling.\n\n", "prompts/v1.txt": "# Prompt v1 — 强调 OCR 准确率\nConvert this page to docling. Extract all text content as accurately as possible.\n\n", "prompts/v2.txt": "# Prompt v2 — 强调阅读顺序 / 表格 / 公式 / 版面结构\nConvert this page to docling. Preserve the reading order, table structure, formulas and layout hierarchy.\n\n", "prompts/v3.txt": "# Prompt v3 — 面向结构化 Document Parsing\nYou are a document parsing engine. Convert this page to the DocTags structured format. Capture every text block, title, table (with full structure), formula (as LaTeX), figure and caption. Keep the correct reading order and include bounding boxes for every element.\n\n"}

for rel, content in FILES.items():
    p = Path('/kaggle/working') / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding='utf-8')
sys.path.insert(0, '/kaggle/working')
print('staged:', sorted(FILES))


In [ ]:
import time, traceback

COMMIT = '193627ae9e97d89188468ed1ee3b7a856ff76044'
EXPECTED_FILES = ["scihub_s12237-014-9873-7.pdf_4.jpg", "PPT_PresentationSpecification_page_025.png", "book_en_\u642c\u4e66\u5320-3473-Reactive Programming with RxJS-2015-\u82f1\u6587\u7248_page_021.png"]
TIMERS = {}
ERRORS = {}

def step(name, fn):
    t = time.time()
    try:
        out = fn()
        TIMERS[name + '_sec'] = round(time.time() - t, 1)
        print('[OK] %s (%.1fs)' % (name, TIMERS[name + '_sec']))
        return out
    except Exception:
        ERRORS[name] = traceback.format_exc()
        print('[FAIL] %s' % name)
        traceback.print_exc()
        return None

Path('/kaggle/working/results').mkdir(parents=True, exist_ok=True)
Path('/kaggle/working/results/heartbeat.txt').write_text('started', encoding='utf-8')


In [ ]:
# 依赖 + 官方评测仓库（锁定 commit 的 tarball）
import subprocess, sys, io, tarfile, urllib.request

def _setup():
    deps = ['docling-core', 'huggingface_hub', 'apted', 'beautifulsoup4', 'evaluate',
            'func-timeout', 'Levenshtein', 'loguru', 'lxml', 'nltk', 'pylatexenc',
            'scipy', 'tabulate', 'pyyaml']
    subprocess.run([sys.executable, '-m', 'pip', 'install'] + deps + ['--quiet'], check=True)
    url = 'https://codeload.github.com/opendatalab/OmniDocBench/tar.gz/' + COMMIT
    req = urllib.request.Request(url, headers={'User-Agent': 'codex-audit'})
    raw = urllib.request.urlopen(req, timeout=300).read()
    base = Path('/kaggle/working/odbrepo')
    base.mkdir(parents=True, exist_ok=True)
    with tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz') as tf:
        tf.extractall(base)
    repo = next(base.glob('OmniDocBench-*'))
    assert (repo / 'pdf_validation.py').is_file(), repo
    # 官方 requires-python 为 >=3.10,<3.12 且锁定旧版依赖（lxml==4.9.1 等无
    # Python 3.12 轮子）。Kaggle 为 3.12，故不执行 pip install -e .，
    # 改为上面已安装的不锁版本依赖，并直接从仓库根目录运行 pdf_validation.py
    # （cwd 即 src/ 的包路径）。pip 明细写入日志备查。
    r = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True, timeout=600)
    pip_log = Path('/kaggle/working/results/benchmark/pip_install.log')
    pip_log.parent.mkdir(parents=True, exist_ok=True)
    pip_log.write_text('pip check rc=' + str(r.returncode) + '\n' + (r.stdout or '')[-2000:] + '\n--- STDERR ---\n' + (r.stderr or '')[-2000:], encoding='utf-8')
    return {'repo': str(repo)}

SETUP = step('setup_repo', _setup)
REPO = Path(SETUP['repo']) if SETUP else None


In [ ]:
# 数据集：标注 JSON + 3 张页面图
from src import data

def _dataset():
    root = data.download_dataset(
        allow_patterns=['OmniDocBench.json', 'images/' + EXPECTED_FILES[0],
                        'images/' + EXPECTED_FILES[1], 'images/' + EXPECTED_FILES[2]],
    )
    ann = data.load_annotations(root)
    sel = data.select_pages(ann, n=3, seed=42)
    ids = [data.sample_id(p) for p in sel]
    print('selected:', ids)
    return {'root': str(root), 'ids': ids}

DATASET = step('dataset', _dataset)


In [ ]:
# 基线推理（3 页，CPU，max_new_tokens=4096）
from src.model import SmolDoclingAdapter, model_summary
from src.inference import run_baseline
from src.config import load_config

cfg = load_config()

def _baseline():
    adapter = SmolDoclingAdapter().load()
    print(model_summary(adapter))
    ann = data.load_annotations(DATASET['root'])
    sel = data.select_pages(ann, n=3, seed=42)
    manifest = run_baseline(
        sel, Path(DATASET['root']), adapter,
        output_dir=Path('/kaggle/working/results/baseline'),
        mode='fast', config=cfg, prompt_id='v0',
        n_pages=3, predict_kwargs={'max_new_tokens': 4096},
    )
    return data.read_json(manifest.parent / 'summary.json')

BASELINE = step('baseline_3pages', _baseline) if DATASET else None
print(BASELINE)


In [ ]:
# 官方评测：GT 子集 + 官方配置 + pdf_validation.py
from src import data, evaluation

def _official():
    ann = data.load_annotations(DATASET['root'])
    base = Path('/kaggle/working/results/baseline')
    bench = Path('/kaggle/working/results/benchmark')
    md_dir = bench / 'pred_md'
    evaluation.export_markdown_predictions(base / 'predictions', Path(DATASET['root']), md_dir)
    gt_json = evaluation.gt_subset_for_predictions(md_dir, ann, bench / 'gt_subset.json')
    yaml_path = evaluation.write_end2end_config(REPO, gt_json, md_dir, bench, include_cdm=False)
    template = cfg['omnidocbench_eval']['end2end_cmd_template']
    cmd = template.replace('{repo}', str(REPO)).replace('{config}', str(yaml_path))
    print('CMD:', cmd)
    log = bench / 'official_end2end.log'
    with open(log, 'w', encoding='utf-8') as f:
        r = subprocess.run(cmd, shell=True, cwd=str(REPO), stdout=f, stderr=subprocess.STDOUT, timeout=3600)
    print('rc =', r.returncode)
    if r.returncode != 0:
        raise RuntimeError('official eval failed, see ' + str(log))
    result_src = REPO / 'result'
    result_dst = Path('/kaggle/working/official_result')
    if result_src.is_dir():
        import shutil as _sh
        if result_dst.exists():
            _sh.rmtree(result_dst)
        _sh.copytree(result_src, result_dst)
    files = sorted(str(p.relative_to(result_dst)) for p in result_dst.rglob('*') if p.is_file())
    print('official result files:', files)
    return {'rc': r.returncode, 'result_files': files}

OFFICIAL = step('official_eval', _official) if (DATASET and REPO and BASELINE) else None


In [ ]:
# 汇总报告（任何失败也写出）
from src import data
import shutil as _sh

G = globals()
meta = {
    'experiment_id': 'official-eval-smoke',
    'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'omnidocbench_commit': COMMIT,
    'dataset_revision': cfg['dataset']['revision'],
    'model_revision': cfg['model']['revision'],
    'pages': G.get('DATASET', {}).get('ids') if G.get('DATASET') else None,
    'baseline': G.get('BASELINE'),
    'official': G.get('OFFICIAL'),
    'errors': ERRORS,
    'timers': TIMERS,
}
data.write_json(meta, Path('/kaggle/working/results/official_eval_metadata.json'))
lines = ['OFFICIAL EVAL SMOKE REPORT']
lines.append('timers=' + json.dumps(TIMERS, ensure_ascii=False))
lines.append('baseline=' + json.dumps(G.get('BASELINE'), ensure_ascii=False))
lines.append('official=' + json.dumps(G.get('OFFICIAL'), ensure_ascii=False))
for name, err in ERRORS.items():
    lines.append('ERROR[%s]:\n%s' % (name, err))
Path('/kaggle/working/results/run_report.txt').write_text('\n'.join(lines), encoding='utf-8')
for _big in ('/kaggle/working/data',):
    _sh.rmtree(_big, ignore_errors=True)
print('\n'.join(lines))
